In [5]:
from google.colab import drive
drive.mount('/content/drive')

# Install YOLO
!pip install -q ultralytics pyyaml

from ultralytics import YOLO
from pathlib import Path
import yaml

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
PROJECT_DIR = Path("/content/drive/MyDrive/Machine_Learning")

DATASET_DIR = PROJECT_DIR / "Dataset/YOLO_Dataset/single_class_road_crossing_960_yolov8"

MODEL_FILE = PROJECT_DIR / "Models" / "yolov8m.pt"

OUTPUT_DIR = PROJECT_DIR / "Experiments/Single_Class/YOLOv8m"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
yaml_dict = {
    "path": str(DATASET_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 1,
    "names": ["person"]
}

yaml_path = DATASET_DIR / "data.yaml"

with open(yaml_path, "w") as f:
    yaml.dump(yaml_dict, f)

In [8]:
last_checkpoint = OUTPUT_DIR / "train" / "weights" / "last.pt"

if last_checkpoint.exists():
    print("Resuming training from:", last_checkpoint)
    model = YOLO(str(last_checkpoint))
    resume = True
else:
    print("Starting new training...")
    model = YOLO(str(MODEL_FILE))
    resume = False

Starting new training...


In [9]:
model.train(
    data=str(yaml_path),

    epochs=50,
    imgsz=960,
    batch=8,

    optimizer="AdamW",
    lr0=5e-4,
    lrf=0.01,
    weight_decay=5e-4,

    dropout=0.05,
    label_smoothing=0.05,
    patience=10,
    cos_lr=True,
    cache=True,
    workers=2,
    amp=True,
    seed=42,

    mosaic=1.0,
    mixup=0.1,
    fliplr=0.5,
    translate=0.1,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    project=str(OUTPUT_DIR),
    name="train",
    exist_ok=True,

    resume=resume,

    save=True,
    save_period=5,
    plots=True
)

best_model = OUTPUT_DIR / "train" / "weights" / "best.pt"

print("\nLoading Best Model...")
model = YOLO(str(best_model))

WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.105 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/Machine_Learning/Dataset/YOLO_Dataset/single_class_road_crossing_960_yolov8/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.05, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=

In [10]:
print("\nValidation Results")
metrics = model.val(
    data=str(yaml_path),
    split="val"
)

print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")
print(f"mAP@50    : {metrics.box.map50:.4f}")
print(f"mAP50-95  : {metrics.box.map:.4f}")



Validation Results
Ultralytics 8.4.105 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.6±0.2 ms, read: 28.7±7.6 MB/s, size: 116.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/Machine_Learning/Dataset/YOLO_Dataset/single_class_road_crossing_960_yolov8/valid/labels.cache... 70 images, 29 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 70/70 21.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.2s/it 6.1s
                   all         70         71      0.853      0.831      0.904      0.735
Speed: 12.6ms preprocess, 59.1ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /content/runs/detect/val
Precision : 0.8528
Recall

In [11]:
print("\nTesting on Test Set")
test_metrics = model.val(
    data=str(yaml_path),
    split="test"
)

print(f"Test Precision : {test_metrics.box.mp:.4f}")
print(f"Test Recall    : {test_metrics.box.mr:.4f}")
print(f"Test mAP@50    : {test_metrics.box.map50:.4f}")
print(f"Test mAP50-95  : {test_metrics.box.map:.4f}")


Testing on Test Set
Ultralytics 8.4.105 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
WARNING ⚠️ val: Slow image access detected (ping: 0.6±0.2 ms, read: 0.3±0.1 MB/s, size: 116.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/Machine_Learning/Dataset/YOLO_Dataset/single_class_road_crossing_960_yolov8/test/labels... 69 images, 29 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 69/69 1.7it/s 40.4s
val: New cache created: /content/drive/MyDrive/Machine_Learning/Dataset/YOLO_Dataset/single_class_road_crossing_960_yolov8/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.5s/it 7.3s
                   all         69         86      0.775       0.76      0.806      0.627
Speed: 14.0ms preprocess, 58.4ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to

### Model Performance Summary

| Metric       | Validation Set | Test Set     |
|:-------------|:---------------|:-------------|
| Precision    | {:.4f}         | {:.4f}       |
| Recall       | {:.4f}         | {:.4f}       |
| mAP@50       | {:.4f}         | {:.4f}       |
| mAP50-95     | {:.4f}         | {:.4f}       |

These results indicate how well the model performed on unseen data (validation and test sets) after training. The mAP (mean Average Precision) scores are key indicators of detection performance across different Intersection over Union (IoU) thresholds.

In [12]:
print(f"### Model Performance Summary\n\n"\
      f"| Metric       | Validation Set | Test Set     |\n"\
      f"|:-------------|:---------------|:-------------|\n"\
      f"| Precision    | {metrics.box.mp:.4f}         | {test_metrics.box.mp:.4f}       |\n"\
      f"| Recall       | {metrics.box.mr:.4f}         | {test_metrics.box.mr:.4f}       |\n"\
      f"| mAP@50       | {metrics.box.map50:.4f}         | {test_metrics.box.map50:.4f}       |\n"\
      f"| mAP50-95     | {metrics.box.map:.4f}         | {test_metrics.box.map:.4f}       |\n\n"\
      f"These results indicate how well the model performed on unseen data (validation and test sets) after training. The mAP (mean Average Precision) scores are key indicators of detection performance across different Intersection over Union (IoU) thresholds.")

### Model Performance Summary

| Metric       | Validation Set | Test Set     |
|:-------------|:---------------|:-------------|
| Precision    | 0.8528         | 0.7748       |
| Recall       | 0.8310         | 0.7600       |
| mAP@50       | 0.9036         | 0.8063       |
| mAP50-95     | 0.7354         | 0.6271       |

These results indicate how well the model performed on unseen data (validation and test sets) after training. The mAP (mean Average Precision) scores are key indicators of detection performance across different Intersection over Union (IoU) thresholds.
